# DS2002 Capstone — EV Charging Station Analytics

**Team Number:** `team-07`

**Team Members:**
- Austin Song (rtx2nb)
- Haero Lee (jva6yw)
- Nate Kim (sax6mw)

**Date:** 2026-03-30

---

## Cloud Setup

Authenticate to GCS, download raw data, and verify your team folder.

In [83]:
# Install the GCS library (run once)
!pip install google-cloud-storage -q

In [84]:
import os
import json
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from datetime import datetime

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
client = storage.Client(project="ds2002-492012")
bucket = client.bucket("ds2002-capstone-sp26-v2")
print("GCS authentication successful.")
print(f"Bucket: {bucket.name}")

GCS authentication successful.
Bucket: ds2002-capstone-sp26-v2


In [85]:
# Download raw data from GCS
TEAM = "team-07"  # <-- change to your team number

files = [
    "raw-data/charging_sessions.csv",
    "raw-data/station_locations.csv",
    "raw-data/vehicle_types.csv",
    "raw-data/grid_operators.csv",
    "raw-data/energy_and_demand.db",
]

os.makedirs("data", exist_ok=True)
for f in files:
    blob = bucket.blob(f)
    local = os.path.join("data", os.path.basename(f))
    blob.download_to_filename(local)
    print(f"Downloaded {f}")

Downloaded raw-data/charging_sessions.csv
Downloaded raw-data/station_locations.csv
Downloaded raw-data/vehicle_types.csv
Downloaded raw-data/grid_operators.csv
Downloaded raw-data/energy_and_demand.db


## Data Loading

In [86]:
sessions = pd.read_csv("data/charging_sessions.csv")
stations = pd.read_csv("data/station_locations.csv")
vehicles = pd.read_csv("data/vehicle_types.csv")
operators = pd.read_csv("data/grid_operators.csv")

conn = sqlite3.connect("data/energy_and_demand.db")
demand = pd.read_sql("SELECT * FROM daily_demand_summary", conn)
grid = pd.read_sql("SELECT * FROM grid_capacity_levels", conn)
conn.close()

print(f"Sessions: {sessions.shape}")
print(f"Stations: {stations.shape}")
print(f"Vehicles: {vehicles.shape}")
print(f"Operators: {operators.shape}")
print(f"Demand:   {demand.shape}")
print(f"Grid:     {grid.shape}")

Sessions: (27451, 11)
Stations: (21, 8)
Vehicles: (42, 6)
Operators: (5, 7)
Demand:   (6570, 7)
Grid:     (1825, 6)


In [87]:
display(stations.head(20))

,station_id,station_name,city,state,zip_code,latitude,longitude,region
0,STN-001,Downtown Charging Hub,Charlottesville,VA,22902,38.0293,-78.4767,Central
1,STN-002,University Ave Station,Charlottesville,VA,22903,38.0336,-78.5080,Central
2,STN-003,Pantops Charging Plaza,Charlottesville,VA,22911,38.0280,-78.4490,East
3,STN-004,Barracks Road Station,Charlottesville,VA,22903,38.0440,-78.5070,West
4,STN-005,Rivanna Station,Charlottesville,Virginia,22911,38.0520,-78.4430,East
5,STN-006,5th Street Hub,Charlottesville,VA,22902,38.0200,-78.4880,Central
6,STN-007,Rio Road Charging,Charlottesville,VA,22901,38.0700,-78.4800,North
7,STN-008,Hydraulic Rd Station,Charlottesville,VA,22901,38.0560,-78.4990,North
8,STN-009,Seminole Trail Plaza,Charlottesville,VA,22901,38.0800,-78.4850,North
9,STN-010,Avon Street Hub,Charlottesville,virginia,22902,38.0100,-78.4750,South


## Data Exploration

Inspect each dataset before cleaning.

In [88]:
dfs = {
    'sessions': sessions,
    'stations': stations,
    'vehicles': vehicles,
    'operators': operators,
    'demand': demand,
    'grid': grid
}

# shapes
print("-----df shapes-----")
for name, df in dfs.items():
    print(f"{name}: {df.shape}")

# dtypes
print("\n-----df dtypes-----")
for name, df in dfs.items():
    print(f"\n--{name}:--")
    for col, dtype in df.dtypes.items():
        print(f"  {col}: {dtype}")

# info
print("\n-----df info-----")
for name, df in dfs.items():
    print(f"\n--{name}:--")
    df.info()

# nulls
print("\n-----null counts-----")
for name, df in dfs.items():
    print(f"\n--{name}:--")
    print(df.isnull().sum().to_string())

-----df shapes-----
sessions: (27451, 11)
stations: (21, 8)
vehicles: (42, 6)
operators: (5, 7)
demand: (6570, 7)
grid: (1825, 6)

-----df dtypes-----

--sessions:--
  session_id: object
  station_id: object
  vehicle_id: object
  session_start: object
  session_end: object
  kwh_delivered: float64
  session_type: object
  cost_usd: object
  payment_method: object
  connector_used: object
  user_id: object

--stations:--
  station_id: object
  station_name: object
  city: object
  state: object
  zip_code: int64
  latitude: float64
  longitude: float64
  region: object

--vehicles:--
  vehicle_id: object
  vehicle_name: object
  vehicle_class: object
  connector_type: object
  battery_kwh: object
  manufacturer: object

--operators:--
  operator_id: object
  operator_name: object
  city: object
  state: object
  avg_daily_capacity_mw: int64
  peak_capacity_mw: object
  cost_per_kwh: float64

--demand:--
  date: object
  station_id: object
  total_sessions: int64
  total_kwh_delivered: 

In [89]:
display(sessions.head(50))

,session_id,station_id,vehicle_id,session_start,session_end,kwh_delivered,session_type,cost_usd,payment_method,connector_used,user_id
0,SES-005321,STN-013,VH-008,2025-03-16 17:06:25,2025-03-16 18:30:25,1.67,Level 1,0.53,credit card,J1772,U-4469
1,SES-021125,STN-012,VH-005,10-19-2025 17:10:08,10-19-2025 18:40:08,13.69,DC Fast Charge,1.92,debit_card,CHAdeMO,U-4434
2,SES-026798,STN-007,VEH#0005,12/30/2025 07:30,12/30/2025 08:21,9.62,Level 1,3.56,Credit Card,CCS,U-6845
3,SES-008299,STN-006,VH-005,04-28-2025 02:35:41,04-28-2025 04:52:41,70.99,DC Fast Charge,26.27,app_wallet,Tesla Supercharger,U-7854
4,SES-018503,STN-012,VH-011,2025-09-14 14:28:19,2025-09-14 16:39:19,10.15,Level 2,2.84,credit card,CCS,U-2903
5,SES-015711,STN-017,VH-004,2025/08/06 11:54,2025/08/06 12:44,7.60,Level 2,2.43,credit_card,CHAdeMO,U-6552
6,SES-009934,STN16,VH-008,2025-05-24 16:18:09,2025-05-24 17:54:09,47.79,DC Fast Charge,8.12,Google Pay,J1772,U-3960
7,SES-019231,STN-005,VH-001,2025-09-24 17:56:46,2025-09-24 18:10:46,52.43,DC Fast Charge,19.4,Credit Card,CCS,U-5016
8,SES-000993,STN-010,V_ford_f-150_lightning,01/16/2025 20:56,01/16/2025 22:49,19.41,Level 2,2.91,credit card,CCS,U-2508
9,SES-004775,STN-005,VH-003,2025-03-11 18:01:00,2025-03-11 18:38:00,12.83,Level 2,5.65,debit_card,Tesla Supercharger,U-1992


## Data Cleaning Pipeline

Document every cleaning step. Show before and after.

In [99]:
# clean VEHICLES
# standardize strings to fill missing battery_kwh
vehicles['vehicle_name_clean'] = vehicles['vehicle_name'].str.lower().str.strip()
vehicles['vehicle_class_clean'] = vehicles['vehicle_class'].str.lower().str.strip()

# create a reference map for battery capacity
cap_map = vehicles.dropna(subset=['battery_kwh']).set_index(['vehicle_name_clean', 'vehicle_class_clean'])['battery_kwh'].to_dict()

# fill missing values using the map
def fill_kwh(row):
    if pd.isna(row['battery_kwh']) or row['battery_kwh'] == '':
        return cap_map.get((row['vehicle_name_clean'], row['vehicle_class_clean']))
    return row['battery_kwh']

vehicles['battery_kwh'] = vehicles.apply(fill_kwh, axis=1)

# filter to only standard IDs (VH-###) AFTER all previous cleaning steps on 'vehicles'
vehicles = vehicles[vehicles['vehicle_id'].str.contains('^VH-', na=False)].copy()
vehicles.drop(columns=['vehicle_name_clean', 'vehicle_class_clean'], inplace=True)

# fix ccs capitalization
vehicles['connector_type'] = vehicles['connector_type'].str.replace('ccs', 'CCS', case=False)

#fix special characters
vehicles['battery_kwh'] = vehicles['battery_kwh'].str.replace(r'\$', '', regex=True)

# fix NaN manufacturers
vehicles['manufacturer'] = vehicles.apply(
    lambda row: row['vehicle_name'].split()[0] if pd.isna(row['manufacturer']) or row['manufacturer'].strip() == '' else row['manufacturer'],
    axis=1
)

# clean STATIONS
# regulate station state
stations = stations[stations['state'].str.upper().str.strip() == 'VA'].copy()

# regulate station ID
stations['station_id'] = stations['station_id'].str.replace(r'[-_]?(\d+)', lambda m: f"-{m.group(1).zfill(3)}", regex=True)

# get rid of nans and dupes
stations.dropna(inplace=True)
stations.drop_duplicates(subset = ["latitude", "longitude"], inplace=True)

# clean OPERATORS

''' GO-NRG is used in other dataframes a fair amount. But because we're missing data, we can't use it'''

null_mask = operators['cost_per_kwh'].isnull()
null_station_ids = operators.loc[null_mask, 'operator_id'].tolist()

operators['peak_capacity_mw'] = operators['peak_capacity_mw'].str.replace(r'[^a-zA-Z0-9]', '', regex=True)
operators['cost_per_kwh'] = operators['cost_per_kwh'].fillna(round(operators['cost_per_kwh'].median(), 2))

# drop any null
operators.dropna(inplace=True)

# clean SESSIONS
sessions = sessions.sort_values('station_id').reset_index(drop=True)
vehicle_id_map = {
    # VEH#000X → VH-00X format
    'VEH#0001': 'VH-001',
    'VEH#0002': 'VH-002',
    'VEH#0003': 'VH-003',
    'VEH#0004': 'VH-004',
    'VEH#0005': 'VH-005',
    'VEH#0006': 'VH-006',
    'VEH#0007': 'VH-007',
    'VEH#0008': 'VH-008',
    'VEH#0009': 'VH-009',
    'VEH#0010': 'VH-010',
    'VEH#0011': 'VH-011',
    'VEH#0012': 'VH-012',
    'VEH#0013': 'VH-013',
    'VEH#0014': 'VH-014',
    # V_name style — match to vehicles_cleaned by name
    'V_tesla_model_3':     'VH-001',
    'V_tesla_model_y':     'VH-002',
    'V_tesla_model_s':     'VH-003',
    'V_chevrolet_bolt_ev': 'VH-004',
    'V_chevrolet_bolt_euv':'VH-005',
    'V_ford_f-150_lightning': 'VH-006',
    'V_hyundai_ioniq_5':   'VH-007',
    'V_bmw_ix':            'VH-008',
    'V_lucid_air':         'VH-009',
    'V_rivian_r1t':        'VH-010',
    'V_ford_mustang_mach-e': 'VH-011',
    'V_nissan_leaf':       'VH-012',
    'V_kia_ev6':           'VH-013',
    'V_volkswagen_id.4':   'VH-014',
}

# standardize vehicle ids
sessions['vehicle_id'] = sessions['vehicle_id'].replace(vehicle_id_map)

#standardize stations IDs
sessions['station_id'] = sessions['station_id'].str.replace(r'(?<=[a-zA-Z])(?=\d)|(?<=\d)(?=[a-zA-Z])', '-', regex=True)
sessions['station_id'] = sessions['station_id'].str.replace(r'-(\d{1,2})$', lambda m: f"-{m.group(1).zfill(3)}", regex=True) #used ai here

#standardize session start and end
sessions['session_start'] = pd.to_datetime(sessions['session_start'], format = 'mixed')
sessions['session_end'] = pd.to_datetime(sessions['session_end'], format = 'mixed')

# clean $
sessions['cost_usd'] = sessions['cost_usd'].astype(str).str.replace(r'\$', '', regex=True)
sessions['cost_usd'] = pd.to_numeric(sessions['cost_usd'], errors='coerce')

# clean payment method
sessions['payment_method'] = sessions['payment_method'].str.lower().str.replace(r'\s', '_', regex=True)

# drop null values. Can't replace them with anything, they're pure inputs from users. Dropped 300 columns of 27,451
sessions.dropna(inplace=True)

In [100]:
display(stations.head(20))

,station_id,station_name,city,state,zip_code,latitude,longitude,region
0,STN-001,Downtown Charging Hub,Charlottesville,VA,22902,38.0293,-78.4767,Central
1,STN-002,University Ave Station,Charlottesville,VA,22903,38.0336,-78.5080,Central
2,STN-003,Pantops Charging Plaza,Charlottesville,VA,22911,38.0280,-78.4490,East
3,STN-004,Barracks Road Station,Charlottesville,VA,22903,38.0440,-78.5070,West
5,STN-006,5th Street Hub,Charlottesville,VA,22902,38.0200,-78.4880,Central
6,STN-007,Rio Road Charging,Charlottesville,VA,22901,38.0700,-78.4800,North
7,STN-008,Hydraulic Rd Station,Charlottesville,VA,22901,38.0560,-78.4990,North
8,STN-009,Seminole Trail Plaza,Charlottesville,VA,22901,38.0800,-78.4850,North
10,STN-011,Fontaine Ave Station,Charlottesville,VA,22903,38.0300,-78.5150,West
12,STN-013,Albemarle Square,Charlottesville,VA,22901,38.0650,-78.4950,North


## External API Integration

Pull weather or energy data and join with your charging data.

In [92]:
# Your API code here


---

## Question 1: Demand Surge Identification

> Which time periods experienced the greatest charging demand surges compared to the baseline?

In [93]:
# Your analysis for Q1


**Findings:** *(write your interpretation here)*

---

## Question 2: The Vehicle Consolidation Problem

> After standardizing all vehicle ID variants, what is the true daily charging volume by vehicle type?

In [94]:
# Your analysis for Q2


**Findings:** *(write your interpretation here)*

---

## Question 3: Weather and Grid Correlation

> How do temperature extremes correlate with daily charging demand and grid load?

In [95]:
# Your analysis for Q3


**Findings:** *(write your interpretation here)*

---

## Question 4: Station-Level Geographic Patterns

> Do all stations experience the same usage patterns?

In [96]:
# Your analysis for Q4


**Findings:** *(write your interpretation here)*

---

## Question 5: The Connector Type Investigation

> Is the CHAdeMO decline real, or a data artifact?

In [97]:
# Your analysis for Q5


**Findings:** *(write your interpretation here)*

---

## Cloud Upload

Upload your cleaned data back to your team folder in GCS.

In [98]:
# Upload cleaned files to GCS
# blob = bucket.blob(f"{TEAM}/cleaned_sessions.csv")
# blob.upload_from_filename("cleaned_sessions.csv")
# print("Uploaded.")


---

## Reflection

### 1. Data Quality Impact
*(your response)*

### 2. Cloud Pipeline Experience
*(your response)*

### 3. ETL Trade-offs
*(your response)*

### 4. Pipeline Trust
*(your response)*

### 5. Team Collaboration
*(your response)*